# Extração independente e auditável — Copa do Mundo FIFA 2026

Este notebook roda **sem repositório, sem upload obrigatório e sem chave de API**.

Ele contém internamente o calendário de 104 partidas, incluindo a correção do chaveamento:

- jogo 94: **Estados Unidos x Bélgica**;
- vitória da Bélgica classifica a seleção;
- jogo 98: **Espanha x Bélgica**.

A execução cria uma pasta isolada com:

- respostas brutas e manifesto SHA-256;
- placares e resultados validados;
- eventos, estatísticas de equipes e jogadores;
- escalações, arbitragem e cobranças de pênaltis;
- relatórios de lacunas, conflitos e qualidade;
- arquivos compatíveis para posterior integração ao repositório;
- ZIP final para download.

> O notebook nunca altera um repositório existente. Toda saída fica dentro de `wc2026_independent_output/`.

## 1. Dependências

In [ ]:
# Instala somente pacotes ausentes.
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "pandas": "pandas>=2.0",
    "requests": "requests>=2.31",
    "bs4": "beautifulsoup4>=4.12",
    "tqdm": "tqdm>=4.66",
}

missing = [pip_name for module, pip_name in REQUIRED_PACKAGES.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

print("Dependências prontas.")

## 2. Configuração independente

In [ ]:
from __future__ import annotations

from collections import defaultdict
from dataclasses import dataclass
from datetime import datetime, timezone
from hashlib import sha256
from io import StringIO
from pathlib import Path
from typing import Any, Iterable, Optional
import json
import math
import os
import random
import re
import shutil
import time
import unicodedata

import pandas as pd
import requests
from bs4 import BeautifulSoup
from IPython.display import display
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_colwidth", 160)

# -------------------------- CONFIGURAÇÃO PRINCIPAL --------------------------
EXECUTE_NETWORK = True
FETCH_MATCH_SUMMARIES = True
ONLY_COMPLETED = True
MAX_SUMMARIES: Optional[int] = None   # use 3 ou 5 para teste rápido
REQUEST_TIMEOUT_SECONDS = 90
MAX_RETRIES = 5
BASE_DELAY_SECONDS = 2.0
MAX_DELAY_SECONDS = 5.0
AUTO_DOWNLOAD_ZIP = True

# Pasta estável. Cada execução cria um subdiretório por RUN_ID.
BASE_DIR = Path("/content") if Path("/content").exists() else Path.cwd()
WORK_ROOT = BASE_DIR / "wc2026_independent_output"
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_AT_UTC = datetime.now(timezone.utc)
RUN_DIR = WORK_ROOT / RUN_ID
RAW_RUN_DIR = RUN_DIR / "raw"
NORMALIZED_DIR = RUN_DIR / "normalized"
REPORT_DIR = RUN_DIR / "reports"
PATCH_DIR = RUN_DIR / "repository_patch"
MANIFEST_PATH = RUN_DIR / "raw_manifest.jsonl"

for directory in [RAW_RUN_DIR, NORMALIZED_DIR, REPORT_DIR, PATCH_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

ESPN_LEAGUE = "fifa.world"
ESPN_SCOREBOARD_URL = f"https://site.api.espn.com/apis/site/v2/sports/soccer/{ESPN_LEAGUE}/scoreboard"
ESPN_SUMMARY_URL = f"https://site.api.espn.com/apis/site/v2/sports/soccer/{ESPN_LEAGUE}/summary"
FIFA_OFFICIAL_URLS = [
    "https://www.fifa.com/en/tournaments/mens/worldcup/canadamexicousa2026/scores-fixtures",
    "https://www.fifa.com/en/tournaments/mens/worldcup/canadamexicousa2026/articles/match-schedule-fixtures-results-teams-stadiums",
]

# Calendário e aliases incorporados para não depender de arquivo externo.
EMBEDDED_CALENDAR_CSV = "jogo,data,diaSemana,horaET,horaLocal,fase,ordemFase,grupo,rodadaGrupo,equipe1,equipe2,confronto,estadio,cidade,pais,status\n1,2026-06-11,Quinta-feira,15:00,13:00,Fase de grupos,1,A,1.0,México,África do Sul,México x África do Sul,Estadio Azteca,Mexico City,México,Finalizado\n2,2026-06-11,Quinta-feira,22:00,20:00,Fase de grupos,1,A,1.0,Coreia do Sul,Tchéquia,Coreia do Sul x Tchéquia,Estadio Akron,Guadalajara,México,Finalizado\n3,2026-06-12,Sexta-feira,15:00,15:00,Fase de grupos,1,B,1.0,Canadá,Bósnia e Herzegovina,Canadá x Bósnia e Herzegovina,BMO Field,Toronto,Canadá,Finalizado\n4,2026-06-12,Sexta-feira,21:00,18:00,Fase de grupos,1,D,1.0,Estados Unidos,Paraguai,Estados Unidos x Paraguai,SoFi Stadium,Los Angeles,Estados Unidos,Finalizado\n5,2026-06-13,Sábado,21:00,21:00,Fase de grupos,1,C,1.0,Haiti,Escócia,Haiti x Escócia,Gillette Stadium,Boston,Estados Unidos,Finalizado\n6,2026-06-13,Sábado,24:00,21:00,Fase de grupos,1,D,1.0,Austrália,Turquia,Austrália x Turquia,BC Place,Vancouver,Canadá,Finalizado\n7,2026-06-13,Sábado,18:00,18:00,Fase de grupos,1,C,1.0,Brasil,Marrocos,Brasil x Marrocos,MetLife Stadium,New York/New Jersey,Estados Unidos,Finalizado\n8,2026-06-13,Sábado,15:00,12:00,Fase de grupos,1,B,1.0,Catar,Suíça,Catar x Suíça,Levi's Stadium,San Francisco Bay Area,Estados Unidos,Finalizado\n9,2026-06-14,Domingo,19:00,19:00,Fase de grupos,1,E,1.0,Costa do Marfim,Equador,Costa do Marfim x Equador,Lincoln Financial Field,Philadelphia,Estados Unidos,Finalizado\n10,2026-06-14,Domingo,13:00,12:00,Fase de grupos,1,E,1.0,Alemanha,Curaçao,Alemanha x Curaçao,NRG Stadium,Houston,Estados Unidos,Finalizado\n11,2026-06-14,Domingo,16:00,15:00,Fase de grupos,1,F,1.0,Países Baixos,Japão,Países Baixos x Japão,AT&T Stadium,Dallas,Estados Unidos,Finalizado\n12,2026-06-14,Domingo,22:00,20:00,Fase de grupos,1,F,1.0,Suécia,Tunísia,Suécia x Tunísia,Estadio BBVA,Monterrey,México,Finalizado\n13,2026-06-15,Segunda-feira,18:00,18:00,Fase de grupos,1,H,1.0,Arábia Saudita,Uruguai,Arábia Saudita x Uruguai,Hard Rock Stadium,Miami,Estados Unidos,Finalizado\n14,2026-06-15,Segunda-feira,12:00,12:00,Fase de grupos,1,H,1.0,Espanha,Cabo Verde,Espanha x Cabo Verde,Mercedes-Benz Stadium,Atlanta,Estados Unidos,Finalizado\n15,2026-06-15,Segunda-feira,21:00,18:00,Fase de grupos,1,G,1.0,Irã,Nova Zelândia,Irã x Nova Zelândia,SoFi Stadium,Los Angeles,Estados Unidos,Finalizado\n16,2026-06-15,Segunda-feira,15:00,12:00,Fase de grupos,1,G,1.0,Bélgica,Egito,Bélgica x Egito,Lumen Field,Seattle,Estados Unidos,Finalizado\n17,2026-06-16,Terça-feira,15:00,15:00,Fase de grupos,1,I,1.0,França,Senegal,França x Senegal,MetLife Stadium,New York/New Jersey,Estados Unidos,Finalizado\n18,2026-06-16,Terça-feira,18:00,18:00,Fase de grupos,1,I,1.0,Iraque,Noruega,Iraque x Noruega,Gillette Stadium,Boston,Estados Unidos,Finalizado\n19,2026-06-16,Terça-feira,21:00,20:00,Fase de grupos,1,J,1.0,Argentina,Argélia,Argentina x Argélia,Arrowhead Stadium,Kansas City,Estados Unidos,Finalizado\n20,2026-06-16,Terça-feira,24:00,21:00,Fase de grupos,1,J,1.0,Áustria,Jordânia,Áustria x Jordânia,Levi's Stadium,San Francisco Bay Area,Estados Unidos,Finalizado\n21,2026-06-17,Quarta-feira,19:00,19:00,Fase de grupos,1,L,1.0,Gana,Panamá,Gana x Panamá,BMO Field,Toronto,Canadá,Finalizado\n22,2026-06-17,Quarta-feira,16:00,15:00,Fase de grupos,1,L,1.0,Inglaterra,Croácia,Inglaterra x Croácia,AT&T Stadium,Dallas,Estados Unidos,Finalizado\n23,2026-06-17,Quarta-feira,13:00,12:00,Fase de grupos,1,K,1.0,Portugal,RD Congo,Portugal x RD Congo,NRG Stadium,Houston,Estados Unidos,Finalizado\n24,2026-06-17,Quarta-feira,22:00,20:00,Fase de grupos,1,K,1.0,Uzbequistão,Colômbia,Uzbequistão x Colômbia,Estadio Azteca,Mexico City,México,Finalizado\n25,2026-06-18,Quinta-feira,12:00,12:00,Fase de grupos,1,A,2.0,Tchéquia,África do Sul,Tchéquia x África do Sul,Mercedes-Benz Stadium,Atlanta,Estados Unidos,Finalizado\n26,2026-06-18,Quinta-feira,15:00,12:00,Fase de grupos,1,B,2.0,Suíça,Bósnia e Herzegovina,Suíça x Bósnia e Herzegovina,SoFi Stadium,Los Angeles,Estados Unidos,Finalizado\n27,2026-06-18,Quinta-feira,18:00,15:00,Fase de grupos,1,B,2.0,Canadá,Catar,Canadá x Catar,BC Place,Vancouver,Canadá,Finalizado\n28,2026-06-18,Quinta-feira,21:00,19:00,Fase de grupos,1,A,2.0,México,Coreia do Sul,México x Coreia do Sul,Estadio Akron,Guadalajara,México,Finalizado\n29,2026-06-19,Sexta-feira,21:00,21:00,Fase de grupos,1,C,2.0,Brasil,Haiti,Brasil x Haiti,Lincoln Financial Field,Philadelphia,Estados Unidos,Finalizado\n30,2026-06-19,Sexta-feira,18:00,18:00,Fase de grupos,1,C,2.0,Escócia,Marrocos,Escócia x Marrocos,Gillette Stadium,Boston,Estados Unidos,Finalizado\n31,2026-06-19,Sexta-feira,23:00,20:00,Fase de grupos,1,D,2.0,Turquia,Paraguai,Turquia x Paraguai,Levi's Stadium,San Francisco Bay Area,Estados Unidos,Finalizado\n32,2026-06-19,Sexta-feira,15:00,12:00,Fase de grupos,1,D,2.0,Estados Unidos,Austrália,Estados Unidos x Austrália,Lumen Field,Seattle,Estados Unidos,Finalizado\n33,2026-06-20,Sábado,16:00,16:00,Fase de grupos,1,E,2.0,Alemanha,Costa do Marfim,Alemanha x Costa do Marfim,BMO Field,Toronto,Canadá,Finalizado\n34,2026-06-20,Sábado,20:00,19:00,Fase de grupos,1,E,2.0,Equador,Curaçao,Equador x Curaçao,Arrowhead Stadium,Kansas City,Estados Unidos,Finalizado\n35,2026-06-20,Sábado,13:00,12:00,Fase de grupos,1,F,2.0,Países Baixos,Suécia,Países Baixos x Suécia,NRG Stadium,Houston,Estados Unidos,Finalizado\n36,2026-06-20,Sábado,24:00,22:00,Fase de grupos,1,F,2.0,Tunísia,Japão,Tunísia x Japão,Estadio BBVA,Monterrey,México,Finalizado\n37,2026-06-21,Domingo,18:00,18:00,Fase de grupos,1,H,2.0,Uruguai,Cabo Verde,Uruguai x Cabo Verde,Hard Rock Stadium,Miami,Estados Unidos,Finalizado\n38,2026-06-21,Domingo,12:00,12:00,Fase de grupos,1,H,2.0,Espanha,Arábia Saudita,Espanha x Arábia Saudita,Mercedes-Benz Stadium,Atlanta,Estados Unidos,Finalizado\n39,2026-06-21,Domingo,15:00,12:00,Fase de grupos,1,G,2.0,Bélgica,Irã,Bélgica x Irã,SoFi Stadium,Los Angeles,Estados Unidos,Finalizado\n40,2026-06-21,Domingo,21:00,18:00,Fase de grupos,1,G,2.0,Nova Zelândia,Egito,Nova Zelândia x Egito,BC Place,Vancouver,Canadá,Finalizado\n41,2026-06-22,Segunda-feira,20:00,20:00,Fase de grupos,1,I,2.0,Noruega,Senegal,Noruega x Senegal,MetLife Stadium,New York/New Jersey,Estados Unidos,Finalizado\n42,2026-06-22,Segunda-feira,17:00,17:00,Fase de grupos,1,I,2.0,França,Iraque,França x Iraque,Lincoln Financial Field,Philadelphia,Estados Unidos,Finalizado\n43,2026-06-22,Segunda-feira,13:00,12:00,Fase de grupos,1,J,2.0,Argentina,Áustria,Argentina x Áustria,AT&T Stadium,Dallas,Estados Unidos,Finalizado\n44,2026-06-22,Segunda-feira,23:00,20:00,Fase de grupos,1,J,2.0,Jordânia,Argélia,Jordânia x Argélia,Levi's Stadium,San Francisco Bay Area,Estados Unidos,Finalizado\n45,2026-06-23,Terça-feira,16:00,16:00,Fase de grupos,1,L,2.0,Inglaterra,Gana,Inglaterra x Gana,Gillette Stadium,Boston,Estados Unidos,Finalizado\n46,2026-06-23,Terça-feira,19:00,19:00,Fase de grupos,1,L,2.0,Panamá,Croácia,Panamá x Croácia,BMO Field,Toronto,Canadá,Finalizado\n47,2026-06-23,Terça-feira,13:00,12:00,Fase de grupos,1,K,2.0,Portugal,Uzbequistão,Portugal x Uzbequistão,NRG Stadium,Houston,Estados Unidos,Finalizado\n48,2026-06-23,Terça-feira,22:00,20:00,Fase de grupos,1,K,2.0,Colômbia,RD Congo,Colômbia x RD Congo,Estadio Akron,Guadalajara,México,Finalizado\n49,2026-06-24,Quarta-feira,18:00,18:00,Fase de grupos,1,C,3.0,Escócia,Brasil,Escócia x Brasil,Hard Rock Stadium,Miami,Estados Unidos,Finalizado\n50,2026-06-24,Quarta-feira,18:00,18:00,Fase de grupos,1,C,3.0,Marrocos,Haiti,Marrocos x Haiti,Mercedes-Benz Stadium,Atlanta,Estados Unidos,Finalizado\n51,2026-06-24,Quarta-feira,15:00,12:00,Fase de grupos,1,B,3.0,Suíça,Canadá,Suíça x Canadá,BC Place,Vancouver,Canadá,Finalizado\n52,2026-06-24,Quarta-feira,15:00,12:00,Fase de grupos,1,B,3.0,Bósnia e Herzegovina,Catar,Bósnia e Herzegovina x Catar,Lumen Field,Seattle,Estados Unidos,Finalizado\n53,2026-06-24,Quarta-feira,21:00,19:00,Fase de grupos,1,A,3.0,Tchéquia,México,Tchéquia x México,Estadio Azteca,Mexico City,México,Finalizado\n54,2026-06-24,Quarta-feira,21:00,19:00,Fase de grupos,1,A,3.0,África do Sul,Coreia do Sul,África do Sul x Coreia do Sul,Estadio BBVA,Monterrey,México,Finalizado\n55,2026-06-25,Quinta-feira,16:00,16:00,Fase de grupos,1,E,3.0,Curaçao,Costa do Marfim,Curaçao x Costa do Marfim,Lincoln Financial Field,Philadelphia,Estados Unidos,Finalizado\n56,2026-06-25,Quinta-feira,16:00,16:00,Fase de grupos,1,E,3.0,Equador,Alemanha,Equador x Alemanha,MetLife Stadium,New York/New Jersey,Estados Unidos,Finalizado\n57,2026-06-25,Quinta-feira,19:00,18:00,Fase de grupos,1,F,3.0,Japão,Suécia,Japão x Suécia,AT&T Stadium,Dallas,Estados Unidos,Finalizado\n58,2026-06-25,Quinta-feira,19:00,18:00,Fase de grupos,1,F,3.0,Tunísia,Países Baixos,Tunísia x Países Baixos,Arrowhead Stadium,Kansas City,Estados Unidos,Finalizado\n59,2026-06-25,Quinta-feira,22:00,19:00,Fase de grupos,1,D,3.0,Turquia,Estados Unidos,Turquia x Estados Unidos,SoFi Stadium,Los Angeles,Estados Unidos,Finalizado\n60,2026-06-25,Quinta-feira,22:00,19:00,Fase de grupos,1,D,3.0,Paraguai,Austrália,Paraguai x Austrália,Levi's Stadium,San Francisco Bay Area,Estados Unidos,Finalizado\n61,2026-06-26,Sexta-feira,15:00,15:00,Fase de grupos,1,I,3.0,Noruega,França,Noruega x França,Gillette Stadium,Boston,Estados Unidos,Finalizado\n62,2026-06-26,Sexta-feira,15:00,15:00,Fase de grupos,1,I,3.0,Senegal,Iraque,Senegal x Iraque,BMO Field,Toronto,Canadá,Finalizado\n63,2026-06-26,Sexta-feira,23:00,20:00,Fase de grupos,1,G,3.0,Egito,Irã,Egito x Irã,Lumen Field,Seattle,Estados Unidos,Finalizado\n64,2026-06-26,Sexta-feira,23:00,20:00,Fase de grupos,1,G,3.0,Nova Zelândia,Bélgica,Nova Zelândia x Bélgica,BC Place,Vancouver,Canadá,Finalizado\n65,2026-06-26,Sexta-feira,20:00,19:00,Fase de grupos,1,H,3.0,Cabo Verde,Arábia Saudita,Cabo Verde x Arábia Saudita,NRG Stadium,Houston,Estados Unidos,Finalizado\n66,2026-06-26,Sexta-feira,20:00,18:00,Fase de grupos,1,H,3.0,Uruguai,Espanha,Uruguai x Espanha,Estadio Akron,Guadalajara,México,Finalizado\n67,2026-06-27,Sábado,17:00,17:00,Fase de grupos,1,L,3.0,Panamá,Inglaterra,Panamá x Inglaterra,MetLife Stadium,New York/New Jersey,Estados Unidos,Finalizado\n68,2026-06-27,Sábado,17:00,17:00,Fase de grupos,1,L,3.0,Croácia,Gana,Croácia x Gana,Lincoln Financial Field,Philadelphia,Estados Unidos,Finalizado\n69,2026-06-27,Sábado,22:00,21:00,Fase de grupos,1,J,3.0,Argélia,Áustria,Argélia x Áustria,Arrowhead Stadium,Kansas City,Estados Unidos,Finalizado\n70,2026-06-27,Sábado,22:00,21:00,Fase de grupos,1,J,3.0,Jordânia,Argentina,Jordânia x Argentina,AT&T Stadium,Dallas,Estados Unidos,Finalizado\n71,2026-06-27,Sábado,19:30,19:30,Fase de grupos,1,K,3.0,Colômbia,Portugal,Colômbia x Portugal,Hard Rock Stadium,Miami,Estados Unidos,Finalizado\n72,2026-06-27,Sábado,19:30,19:30,Fase de grupos,1,K,3.0,RD Congo,Uzbequistão,RD Congo x Uzbequistão,Mercedes-Benz Stadium,Atlanta,Estados Unidos,Finalizado\n73,2026-06-28,Domingo,15:00,12:00,16 avos de final,2,,0.0,África do Sul,Canadá,África do Sul x Canadá,SoFi Stadium,Los Angeles,Estados Unidos,Finalizado\n74,2026-06-29,Segunda-feira,16:30,16:30,16 avos de final,2,,0.0,Alemanha,Paraguai,Alemanha x Paraguai,Gillette Stadium,Boston,Estados Unidos,Finalizado\n75,2026-06-29,Segunda-feira,21:00,19:00,16 avos de final,2,,0.0,Países Baixos,Marrocos,Países Baixos x Marrocos,Estadio BBVA,Monterrey,México,Finalizado\n76,2026-06-29,Segunda-feira,13:00,12:00,16 avos de final,2,,0.0,Brasil,Japão,Brasil x Japão,NRG Stadium,Houston,Estados Unidos,Finalizado\n77,2026-06-30,Terça-feira,17:00,17:00,16 avos de final,2,,0.0,França,Suécia,França x Suécia,MetLife Stadium,New York/New Jersey,Estados Unidos,Finalizado\n78,2026-06-30,Terça-feira,13:00,12:00,16 avos de final,2,,0.0,Costa do Marfim,Noruega,Costa do Marfim x Noruega,AT&T Stadium,Dallas,Estados Unidos,Finalizado\n79,2026-06-30,Terça-feira,21:00,19:00,16 avos de final,2,,0.0,México,Equador,México x Equador,Estadio Azteca,Mexico City,México,Finalizado\n80,2026-07-01,Quarta-feira,12:00,12:00,16 avos de final,2,,0.0,Inglaterra,RD Congo,Inglaterra x RD Congo,Mercedes-Benz Stadium,Atlanta,Estados Unidos,Finalizado\n81,2026-07-01,Quarta-feira,20:00,17:00,16 avos de final,2,,0.0,Estados Unidos,Bósnia e Herzegovina,Estados Unidos x Bósnia e Herzegovina,Levi's Stadium,San Francisco Bay Area,Estados Unidos,Finalizado\n82,2026-07-01,Quarta-feira,16:00,13:00,16 avos de final,2,,0.0,Bélgica,Senegal,Bélgica x Senegal,Lumen Field,Seattle,Estados Unidos,Finalizado\n83,2026-07-02,Quinta-feira,19:00,19:00,16 avos de final,2,,0.0,Portugal,Croácia,Portugal x Croácia,BMO Field,Toronto,Canadá,Finalizado\n84,2026-07-02,Quinta-feira,15:00,12:00,16 avos de final,2,,0.0,Espanha,Áustria,Espanha x Áustria,SoFi Stadium,Los Angeles,Estados Unidos,Finalizado\n85,2026-07-02,Quinta-feira,23:00,20:00,16 avos de final,2,,0.0,Suíça,Argélia,Suíça x Argélia,BC Place,Vancouver,Canadá,Finalizado\n86,2026-07-03,Sexta-feira,18:00,18:00,16 avos de final,2,,0.0,Argentina,Cabo Verde,Argentina x Cabo Verde,Hard Rock Stadium,Miami,Estados Unidos,Finalizado\n87,2026-07-03,Sexta-feira,21:30,20:30,16 avos de final,2,,0.0,Colômbia,Gana,Colômbia x Gana,Arrowhead Stadium,Kansas City,Estados Unidos,Finalizado\n88,2026-07-03,Sexta-feira,14:00,13:00,16 avos de final,2,,0.0,Austrália,Egito,Austrália x Egito,AT&T Stadium,Dallas,Estados Unidos,Finalizado\n89,2026-07-04,Sábado,17:00,17:00,Oitavas de final,3,,0.0,Canadá,Marrocos,Canadá x Marrocos,Lincoln Financial Field,Philadelphia,Estados Unidos,Finalizado\n90,2026-07-04,Sábado,13:00,12:00,Oitavas de final,3,,0.0,Paraguai,França,Paraguai x França,NRG Stadium,Houston,Estados Unidos,Finalizado\n91,2026-07-05,Domingo,16:00,16:00,Oitavas de final,3,,0.0,Brasil,Noruega,Brasil x Noruega,MetLife Stadium,New York/New Jersey,Estados Unidos,Finalizado\n92,2026-07-05,Domingo,20:00,18:00,Oitavas de final,3,,0.0,México,Inglaterra,México x Inglaterra,Estadio Azteca,Mexico City,México,Finalizado\n93,2026-07-06,Segunda-feira,15:00,14:00,Oitavas de final,3,,0.0,Portugal,Espanha,Portugal x Espanha,AT&T Stadium,Dallas,Estados Unidos,Finalizado\n94,2026-07-06,Segunda-feira,20:00,17:00,Oitavas de final,3,,0.0,Estados Unidos,Bélgica,Estados Unidos x Bélgica,Lumen Field,Seattle,Estados Unidos,Finalizado\n95,2026-07-07,Terça-feira,12:00,12:00,Oitavas de final,3,,0.0,Argentina,Egito,Argentina x Egito,Mercedes-Benz Stadium,Atlanta,Estados Unidos,Finalizado\n96,2026-07-07,Terça-feira,16:00,13:00,Oitavas de final,3,,0.0,Suíça,Colômbia,Suíça x Colômbia,BC Place,Vancouver,Canadá,Finalizado\n97,2026-07-09,Quinta-feira,16:00,16:00,Quartas de final,4,,0.0,Marrocos,França,Marrocos x França,Gillette Stadium,Boston,Estados Unidos,Finalizado\n98,2026-07-10,Sexta-feira,15:00,12:00,Quartas de final,4,,0.0,Espanha,Bélgica,Espanha x Bélgica,SoFi Stadium,Los Angeles,Estados Unidos,Finalizado\n99,2026-07-11,Sábado,17:00,17:00,Quartas de final,4,,0.0,Noruega,Inglaterra,Noruega x Inglaterra,Hard Rock Stadium,Miami,Estados Unidos,Finalizado\n100,2026-07-11,Sábado,21:00,20:00,Quartas de final,4,,0.0,Suíça,Argentina,Suíça x Argentina,Arrowhead Stadium,Kansas City,Estados Unidos,Finalizado\n101,2026-07-14,Terça-feira,15:00,14:00,Semifinais,5,,0.0,França,Espanha,França x Espanha,AT&T Stadium,Dallas,Estados Unidos,Finalizado\n102,2026-07-15,Quarta-feira,15:00,15:00,Semifinais,5,,0.0,Inglaterra,Argentina,Inglaterra x Argentina,Mercedes-Benz Stadium,Atlanta,Estados Unidos,Finalizado\n103,2026-07-18,Sábado,17:00,17:00,Disputa de 3º lugar,6,,0.0,França,Inglaterra,França x Inglaterra,Hard Rock Stadium,Miami,Estados Unidos,Finalizado\n104,2026-07-19,Domingo,15:00,15:00,Final,7,,0.0,Espanha,Argentina,Espanha x Argentina,MetLife Stadium,New York/New Jersey,Estados Unidos,Finalizado\n"
EMBEDDED_ALIASES_CSV = "repo_team,alias\nMéxico,Mexico\nÁfrica do Sul,South Africa\nCoreia do Sul,South Korea\nCoreia do Sul,Republic of Korea\nTchéquia,Czechia\nTchéquia,Czech Republic\nCanadá,Canada\nBósnia e Herzegovina,Bosnia and Herzegovina\nBósnia e Herzegovina,Bosnia-Herzegovina\nEstados Unidos,United States\nEstados Unidos,USA\nParaguai,Paraguay\nCatar,Qatar\nSuíça,Switzerland\nBrasil,Brazil\nMarrocos,Morocco\nHaiti,Haiti\nEscócia,Scotland\nAustrália,Australia\nTurquia,Turkey\nAlemanha,Germany\nCuraçao,Curacao\nCuraçao,Curaçao\nPaíses Baixos,Netherlands\nPaíses Baixos,Holland\nJapão,Japan\nArgentina,Argentina\nArgélia,Algeria\nÁustria,Austria\nBélgica,Belgium\nCabo Verde,Cape Verde\nColômbia,Colombia\nCosta do Marfim,Ivory Coast\nCosta do Marfim,Cote d'Ivoire\nCosta do Marfim,Côte d'Ivoire\nCroácia,Croatia\nEgito,Egypt\nEquador,Ecuador\nEspanha,Spain\nFrança,France\nGana,Ghana\nInglaterra,England\nIraque,Iraq\nIrã,Iran\nIrã,IR Iran\nJordânia,Jordan\nNoruega,Norway\nNova Zelândia,New Zealand\nPanamá,Panama\nPortugal,Portugal\nRD Congo,DR Congo\nRD Congo,Congo DR\nRD Congo,Democratic Republic of Congo\nSenegal,Senegal\nSuécia,Sweden\nTunísia,Tunisia\nUruguai,Uruguay\nUzbequistão,Uzbekistan\nArábia Saudita,Saudi Arabia\n"

print(f"RUN_ID: {RUN_ID}")
print(f"Saídas: {RUN_DIR}")
print(f"Rede habilitada: {EXECUTE_NETWORK}")

## 3. Calendário incorporado e normalização de nomes

In [ ]:
def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def normalize_text(value: Any) -> str:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return ""
    value = unicodedata.normalize("NFD", str(value).strip().lower())
    value = "".join(ch for ch in value if unicodedata.category(ch) != "Mn")
    value = re.sub(r"[^a-z0-9]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()


def safe_int(value: Any) -> Optional[int]:
    if value is None or value == "":
        return None
    if isinstance(value, dict):
        value = value.get("value", value.get("displayValue"))
    try:
        return int(float(str(value).replace(",", ".")))
    except (TypeError, ValueError):
        match = re.search(r"-?\d+", str(value))
        return int(match.group()) if match else None


def safe_float(value: Any) -> Optional[float]:
    if value is None or value == "":
        return None
    if isinstance(value, dict):
        value = value.get("value", value.get("displayValue"))
    try:
        return float(str(value).replace("%", "").replace(",", "."))
    except (TypeError, ValueError):
        return None


def first_nonempty(*values: Any) -> Any:
    for value in values:
        if value is not None and str(value).strip() not in {"", "nan", "None", "<NA>"}:
            return value
    return None


def nested_get(obj: dict, path: Iterable[str], default=None):
    current = obj
    for key in path:
        if not isinstance(current, dict) or key not in current:
            return default
        current = current[key]
    return current


def write_csv(frame: pd.DataFrame, path: Path, sep: str = ",") -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False, encoding="utf-8-sig", sep=sep)


def dataframe_records(frame: pd.DataFrame) -> list[dict]:
    if frame.empty:
        return []
    clean = frame.copy().astype(object)
    clean = clean.where(pd.notna(clean), None)
    return clean.to_dict("records")


repo_matches = pd.read_csv(StringIO(EMBEDDED_CALENDAR_CSV), encoding="utf-8-sig")
repo_matches["jogo"] = pd.to_numeric(repo_matches["jogo"], errors="coerce").astype("Int64")
repo_matches["data"] = pd.to_datetime(repo_matches["data"], errors="coerce").dt.date.astype("string")

alias_df = pd.read_csv(StringIO(EMBEDDED_ALIASES_CSV), encoding="utf-8-sig")
alias_to_repo: dict[str, str] = {}
repo_to_variants: dict[str, set[str]] = defaultdict(set)

for team in sorted(set(repo_matches["equipe1"].dropna()) | set(repo_matches["equipe2"].dropna())):
    alias_to_repo[normalize_text(team)] = team
    repo_to_variants[team].add(team)

for _, row in alias_df.iterrows():
    repo_team, alias = str(row["repo_team"]).strip(), str(row["alias"]).strip()
    alias_to_repo[normalize_text(alias)] = repo_team
    alias_to_repo[normalize_text(repo_team)] = repo_team
    repo_to_variants[repo_team].update({repo_team, alias})

manual_aliases = {
    "korea republic": "Coreia do Sul",
    "republic of korea": "Coreia do Sul",
    "south korea": "Coreia do Sul",
    "iran": "Irã",
    "ir iran": "Irã",
    "usa": "Estados Unidos",
    "united states": "Estados Unidos",
    "united states of america": "Estados Unidos",
    "cote d ivoire": "Costa do Marfim",
    "ivory coast": "Costa do Marfim",
    "congo dr": "RD Congo",
    "dr congo": "RD Congo",
    "democratic republic of congo": "RD Congo",
    "netherlands": "Países Baixos",
    "turkiye": "Turquia",
    "czech republic": "Tchéquia",
    "czechia": "Tchéquia",
    "bosnia and herzegovina": "Bósnia e Herzegovina",
    "saudi arabia": "Arábia Saudita",
}
for alias, team in manual_aliases.items():
    alias_to_repo[normalize_text(alias)] = team
    repo_to_variants[team].add(alias)


def to_repo_team(name: Any) -> str:
    normalized = normalize_text(name)
    return alias_to_repo.get(normalized, str(name).strip() if name is not None else "")


write_csv(repo_matches, NORMALIZED_DIR / "calendar_seed.csv")
write_csv(alias_df, NORMALIZED_DIR / "team_name_aliases.csv")

print(f"Calendário incorporado: {len(repo_matches)} jogos")
print(f"Aliases: {len(alias_to_repo)}")
display(repo_matches.tail(12))

## 4. Cliente HTTP auditável

In [ ]:
@dataclass
class FetchResult:
    url: str
    status_code: Optional[int]
    content_type: str
    content: bytes
    collected_at: str
    sha256: str
    path: Optional[Path]
    error: Optional[str] = None

    def json(self) -> dict:
        return json.loads(self.content.decode("utf-8-sig"))

    def text(self) -> str:
        return self.content.decode("utf-8", errors="replace")


class AuditedHTTPClient:
    def __init__(self) -> None:
        self.session = requests.Session()
        # Não usa proxy definido no ambiente.
        self.session.trust_env = False
        self.session.headers.update({
            "User-Agent": "wc2026-independent-research-notebook/3.0",
            "Accept-Language": "en-US,en;q=0.9,pt-BR;q=0.8",
        })

    def _manifest(self, source: str, logical_name: str, result: FetchResult) -> None:
        row = {
            "run_id": RUN_ID,
            "source": source,
            "logical_name": logical_name,
            "path": str(result.path.relative_to(RUN_DIR)) if result.path else None,
            "url": result.url,
            "status_code": result.status_code,
            "content_type": result.content_type,
            "collected_at": result.collected_at,
            "sha256": result.sha256,
            "error": result.error,
        }
        with MANIFEST_PATH.open("a", encoding="utf-8") as handle:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")

    def _save(self, source: str, logical_name: str, url: str, response: requests.Response) -> FetchResult:
        collected_at = utc_now_iso()
        digest = sha256(response.content).hexdigest()
        content_type = response.headers.get("content-type", "application/octet-stream")
        ext = ".json" if "json" in content_type else ".html" if "html" in content_type else ".bin"
        source_dir = RAW_RUN_DIR / normalize_text(source).replace(" ", "_")
        source_dir.mkdir(parents=True, exist_ok=True)
        filename = f"{normalize_text(logical_name).replace(' ', '_')[:100]}__{digest[:16]}{ext}"
        path = source_dir / filename
        path.write_bytes(response.content)
        result = FetchResult(url, response.status_code, content_type, response.content, collected_at, digest, path)
        self._manifest(source, logical_name, result)
        return result

    def get(self, source: str, logical_name: str, url: str, params: Optional[dict] = None) -> FetchResult:
        prepared = requests.Request("GET", url, params=params).prepare()
        final_url = prepared.url or url
        last_error: Optional[str] = None

        for attempt in range(1, MAX_RETRIES + 1):
            try:
                response = self.session.get(url, params=params, timeout=REQUEST_TIMEOUT_SECONDS)
                if response.status_code == 429 or response.status_code >= 500:
                    retry_after = safe_float(response.headers.get("Retry-After"))
                    wait = retry_after or min(60.0, 2 ** (attempt - 1) + random.uniform(0.5, 2.0))
                    if attempt < MAX_RETRIES:
                        time.sleep(wait)
                        continue
                response.raise_for_status()
                result = self._save(source, logical_name, final_url, response)
                time.sleep(random.uniform(BASE_DELAY_SECONDS, MAX_DELAY_SECONDS))
                return result
            except (requests.RequestException, ValueError) as exc:
                last_error = str(exc)
                if attempt < MAX_RETRIES:
                    time.sleep(min(60.0, 2 ** (attempt - 1) + random.uniform(0.5, 2.0)))

        failed = FetchResult(final_url, None, "", b"", utc_now_iso(), "", None, last_error or "unknown_error")
        self._manifest(source, logical_name, failed)
        raise RuntimeError(f"Falha após {MAX_RETRIES} tentativas: {final_url} | {last_error}")


http = AuditedHTTPClient()
print(f"Manifesto: {MANIFEST_PATH}")

## 5. Scoreboard, placares e metadados de partidas

In [ ]:
def competitor_by_home_away(competitors: list[dict], home_away: str) -> dict:
    return next((c for c in competitors if c.get("homeAway") == home_away), {})


def parse_linescores(competitor: dict) -> tuple[Optional[int], Optional[int], dict[str, Any]]:
    periods: dict[str, Any] = {}
    regulation = 0
    regulation_known = False
    extra = 0
    extra_known = False
    for idx, item in enumerate(competitor.get("linescores") or [], start=1):
        period = safe_int(item.get("period")) or idx
        value = safe_int(first_nonempty(item.get("value"), item.get("displayValue")))
        label = first_nonempty(item.get("periodName"), item.get("name"), f"period_{period}")
        periods[str(label)] = value
        if value is not None and period <= 2:
            regulation += value
            regulation_known = True
        elif value is not None and 2 < period <= 4:
            extra += value
            extra_known = True
    return (
        regulation if regulation_known else None,
        regulation + extra if regulation_known and extra_known else None,
        periods,
    )


def extract_penalty_score(competitor: dict) -> Optional[int]:
    for key in ["shootoutScore", "penaltyScore", "penalties", "shootout"]:
        parsed = safe_int(competitor.get(key))
        if parsed is not None:
            return parsed
    for item in competitor.get("linescores") or []:
        label = normalize_text(first_nonempty(item.get("periodName"), item.get("name"), nested_get(item, ["type", "text"])))
        if "pen" in label or "shootout" in label:
            return safe_int(first_nonempty(item.get("value"), item.get("displayValue")))
    return None


def parse_espn_scoreboard(payload: dict, fetch_meta: FetchResult) -> pd.DataFrame:
    rows = []
    for event in payload.get("events") or []:
        competition = (event.get("competitions") or [{}])[0]
        competitors = competition.get("competitors") or []
        home = competitor_by_home_away(competitors, "home") or (competitors[0] if competitors else {})
        away = competitor_by_home_away(competitors, "away") or (competitors[1] if len(competitors) > 1 else {})
        home_team_obj, away_team_obj = home.get("team") or {}, away.get("team") or {}
        status_type = nested_get(event, ["status", "type"], {}) or {}
        venue = competition.get("venue") or {}
        address = venue.get("address") or {}
        notes = competition.get("notes") or []
        phase = first_nonempty(
            notes[0].get("headline") if notes and isinstance(notes[0], dict) else None,
            nested_get(competition, ["type", "text"]),
            nested_get(event, ["season", "type", "name"]),
        )

        home_90, home_120, home_periods = parse_linescores(home)
        away_90, away_120, away_periods = parse_linescores(away)

        rows.append({
            "espn_event_id": str(event.get("id", "")),
            "date": str(event.get("date", ""))[:10],
            "datetime_utc": event.get("date"),
            "name": event.get("name"),
            "short_name": event.get("shortName"),
            "phase_espn": phase,
            "home_team_espn": first_nonempty(home_team_obj.get("displayName"), home_team_obj.get("name")),
            "away_team_espn": first_nonempty(away_team_obj.get("displayName"), away_team_obj.get("name")),
            "home_team": to_repo_team(first_nonempty(home_team_obj.get("displayName"), home_team_obj.get("name"))),
            "away_team": to_repo_team(first_nonempty(away_team_obj.get("displayName"), away_team_obj.get("name"))),
            "home_team_id": home_team_obj.get("id"),
            "away_team_id": away_team_obj.get("id"),
            "home_score": safe_int(home.get("score")),
            "away_score": safe_int(away.get("score")),
            "home_score_90": home_90,
            "away_score_90": away_90,
            "home_score_120": home_120,
            "away_score_120": away_120,
            "home_penalty_score": extract_penalty_score(home),
            "away_penalty_score": extract_penalty_score(away),
            "home_period_scores_json": json.dumps(home_periods, ensure_ascii=False),
            "away_period_scores_json": json.dumps(away_periods, ensure_ascii=False),
            "status_name": status_type.get("name"),
            "status_description": status_type.get("description"),
            "status_detail": status_type.get("detail"),
            "completed": bool(status_type.get("completed", False)),
            "winner_home": bool(home.get("winner", False)),
            "winner_away": bool(away.get("winner", False)),
            "venue": first_nonempty(venue.get("fullName"), venue.get("name")),
            "city": first_nonempty(address.get("city"), address.get("state")),
            "country": address.get("country"),
            "attendance": competition.get("attendance"),
            "neutral_site": competition.get("neutralSite"),
            "source_url": fetch_meta.url,
            "collected_at": fetch_meta.collected_at,
            "source_sha256": fetch_meta.sha256,
        })
    return pd.DataFrame(rows)


def load_scoreboard() -> pd.DataFrame:
    if not EXECUTE_NETWORK:
        raise RuntimeError("EXECUTE_NETWORK=False. Ative a rede para a execução independente.")
    result = http.get(
        source="espn_scoreboard",
        logical_name="world_cup_2026_scoreboard",
        url=ESPN_SCOREBOARD_URL,
        params={"dates": "2026", "limit": 500},
    )
    frame = parse_espn_scoreboard(result.json(), result)
    if frame.empty:
        raise RuntimeError("A resposta do scoreboard não trouxe eventos.")
    frame = frame.drop_duplicates("espn_event_id", keep="last").reset_index(drop=True)
    return frame


espn_matches = load_scoreboard()
print(f"Eventos encontrados: {len(espn_matches)}")
print(espn_matches["completed"].value_counts(dropna=False))
display(espn_matches.head())

## 6. Mapeamento para os 104 jogos e propagação do chaveamento

In [ ]:
def team_pair(team1: Any, team2: Any) -> frozenset[str]:
    return frozenset({normalize_text(to_repo_team(team1)), normalize_text(to_repo_team(team2))})


def date_distance_days(a: Any, b: Any) -> Optional[int]:
    try:
        return abs((pd.to_datetime(a).date() - pd.to_datetime(b).date()).days)
    except Exception:
        return None


KNOCKOUT_SOURCE_SLOTS = {
    89: (73, 75), 90: (74, 77), 91: (76, 78), 92: (79, 80),
    93: (83, 84), 94: (81, 82), 95: (86, 88), 96: (85, 87),
    97: (89, 90), 98: (93, 94), 99: (91, 92), 100: (96, 95),
    101: (97, 98), 102: (99, 100), 104: (101, 102),
}
THIRD_PLACE_SOURCE = (101, 102)


def event_winner_and_loser(event: pd.Series, home: str, away: str) -> tuple[Optional[str], Optional[str]]:
    winner = None
    if bool(event.get("winner_home", False)):
        winner = home
    elif bool(event.get("winner_away", False)):
        winner = away
    else:
        hp, ap = safe_int(event.get("home_penalty_score")), safe_int(event.get("away_penalty_score"))
        hs, a_s = safe_int(event.get("home_score")), safe_int(event.get("away_score"))
        if hp is not None and ap is not None and hp != ap:
            winner = home if hp > ap else away
        elif hs is not None and a_s is not None and hs != a_s:
            winner = home if hs > a_s else away
    if winner is None:
        return None, None
    loser = away if normalize_text(winner) == normalize_text(home) else home
    return winner, loser


def map_events_to_calendar(events: pd.DataFrame, schedule: pd.DataFrame) -> pd.DataFrame:
    mapped = events.sort_values(["date", "datetime_utc", "espn_event_id"], na_position="last").copy()
    mapped["_original_order"] = range(len(mapped))
    schedule_original = schedule.copy()
    schedule_work = schedule.copy()
    winners: dict[int, str] = {}
    losers: dict[int, str] = {}
    used_games: set[int] = set()

    def propagate_bracket() -> None:
        changed = True
        while changed:
            changed = False
            for target, (source1, source2) in KNOCKOUT_SOURCE_SLOTS.items():
                if source1 not in winners or source2 not in winners:
                    continue
                mask = schedule_work["jogo"].eq(target)
                if not mask.any():
                    continue
                new1, new2 = winners[source1], winners[source2]
                old1 = str(schedule_work.loc[mask, "equipe1"].iloc[0])
                old2 = str(schedule_work.loc[mask, "equipe2"].iloc[0])
                if team_pair(old1, old2) != team_pair(new1, new2):
                    schedule_work.loc[mask, "equipe1"] = new1
                    schedule_work.loc[mask, "equipe2"] = new2
                    schedule_work.loc[mask, "confronto"] = f"{new1} x {new2}"
                    changed = True
            if all(source in losers for source in THIRD_PLACE_SOURCE):
                mask = schedule_work["jogo"].eq(103)
                if mask.any():
                    new1, new2 = losers[101], losers[102]
                    old1 = str(schedule_work.loc[mask, "equipe1"].iloc[0])
                    old2 = str(schedule_work.loc[mask, "equipe2"].iloc[0])
                    if team_pair(old1, old2) != team_pair(new1, new2):
                        schedule_work.loc[mask, "equipe1"] = new1
                        schedule_work.loc[mask, "equipe2"] = new2
                        schedule_work.loc[mask, "confronto"] = f"{new1} x {new2}"
                        changed = True

    assignments = []
    for _, event in mapped.iterrows():
        home = to_repo_team(event.get("home_team"))
        away = to_repo_team(event.get("away_team"))
        pair = team_pair(home, away)
        event_date = event.get("date")
        selected = None
        confidence = "unmapped"
        reason = ""

        candidates = schedule_work[
            schedule_work.apply(lambda r: team_pair(r.get("equipe1"), r.get("equipe2")) == pair, axis=1)
            & ~schedule_work["jogo"].astype(int).isin(used_games)
        ]
        near = candidates[candidates["data"].map(lambda value: (date_distance_days(value, event_date) if date_distance_days(value, event_date) is not None else 99) <= 1)]
        if len(near) == 1:
            selected, confidence, reason = near.iloc[0].to_dict(), "date_team", "Mesma dupla e data ±1 dia"
        elif len(candidates) == 1:
            selected, confidence, reason = candidates.iloc[0].to_dict(), "team_only", "Dupla única no calendário"

        if selected is None:
            same_date = schedule_work[
                schedule_work["data"].map(lambda value: (date_distance_days(value, event_date) if date_distance_days(value, event_date) is not None else 99) <= 1)
                & ~schedule_work["jogo"].astype(int).isin(used_games)
            ]
            phase_norm = normalize_text(event.get("phase_espn"))
            phase_candidates = same_date[
                same_date["fase"].map(lambda value: normalize_text(value) in phase_norm or phase_norm in normalize_text(value))
            ] if phase_norm else same_date.iloc[0:0]
            pool = phase_candidates if len(phase_candidates) else same_date
            if len(pool) == 1:
                selected, confidence, reason = pool.iloc[0].to_dict(), "date_slot_review", "Único slot disponível na data"

        if selected is not None:
            game = int(selected["jogo"])
            used_games.add(game)
            winner, loser = event_winner_and_loser(event, home, away)
            if winner:
                winners[game] = winner
                losers[game] = loser
                propagate_bracket()

            original = schedule_original[schedule_original["jogo"].eq(game)].iloc[0].to_dict()
            participant_changed = team_pair(original.get("equipe1"), original.get("equipe2")) != pair
            assignment = {
                "jogo": game,
                "calendar_equipe1_original": original.get("equipe1"),
                "calendar_equipe2_original": original.get("equipe2"),
                "calendar_equipe1": selected.get("equipe1"),
                "calendar_equipe2": selected.get("equipe2"),
                "calendar_data": selected.get("data"),
                "calendar_fase": selected.get("fase"),
                "mapping_confidence": confidence,
                "mapping_reason": reason,
                "participant_change_required": participant_changed,
            }
        else:
            assignment = {
                "jogo": pd.NA,
                "calendar_equipe1_original": pd.NA,
                "calendar_equipe2_original": pd.NA,
                "calendar_equipe1": pd.NA,
                "calendar_equipe2": pd.NA,
                "calendar_data": pd.NA,
                "calendar_fase": pd.NA,
                "mapping_confidence": "unmapped",
                "mapping_reason": "Nenhum slot único encontrado",
                "participant_change_required": False,
            }
        assignments.append(assignment)

    result = pd.concat([mapped.reset_index(drop=True), pd.DataFrame(assignments)], axis=1)
    result = result.sort_values("_original_order").drop(columns="_original_order").reset_index(drop=True)
    result["jogo"] = pd.to_numeric(result["jogo"], errors="coerce").astype("Int64")
    return result


espn_matches = map_events_to_calendar(espn_matches, repo_matches)
mapping_report = espn_matches[[
    "jogo", "espn_event_id", "date", "home_team", "away_team",
    "calendar_equipe1_original", "calendar_equipe2_original",
    "mapping_confidence", "mapping_reason", "participant_change_required", "completed"
]].sort_values(["jogo", "date"], na_position="last")

print(mapping_report["mapping_confidence"].value_counts(dropna=False))
display(mapping_report.tail(20))

## 7. Resumos: conferência de placar, eventos, desempenho, jogadores, arbitragem e pênaltis

In [ ]:
def stat_name(value: Any) -> str:
    return normalize_text(value).replace(" ", "_")


def summary_competition(payload: dict) -> dict:
    return ((payload.get("header") or {}).get("competitions") or [{}])[0]


def parse_summary_meta(payload: dict, game: Optional[int], event_id: str) -> pd.DataFrame:
    comp = summary_competition(payload)
    competitors = comp.get("competitors") or []
    home = competitor_by_home_away(competitors, "home") or (competitors[0] if competitors else {})
    away = competitor_by_home_away(competitors, "away") or (competitors[1] if len(competitors) > 1 else {})
    home_team_obj, away_team_obj = home.get("team") or {}, away.get("team") or {}
    status_type = nested_get(comp, ["status", "type"], {}) or {}
    return pd.DataFrame([{
        "jogo": game,
        "espn_event_id": event_id,
        "summary_home_team": to_repo_team(first_nonempty(home_team_obj.get("displayName"), home_team_obj.get("name"))),
        "summary_away_team": to_repo_team(first_nonempty(away_team_obj.get("displayName"), away_team_obj.get("name"))),
        "summary_home_score": safe_int(home.get("score")),
        "summary_away_score": safe_int(away.get("score")),
        "summary_home_penalty_score": extract_penalty_score(home),
        "summary_away_penalty_score": extract_penalty_score(away),
        "summary_winner_home": bool(home.get("winner", False)),
        "summary_winner_away": bool(away.get("winner", False)),
        "summary_completed": bool(status_type.get("completed", False)),
        "summary_status": first_nonempty(status_type.get("detail"), status_type.get("description"), status_type.get("name")),
        "source_url": f"{ESPN_SUMMARY_URL}?event={event_id}",
        "collected_at": utc_now_iso(),
    }])


def parse_summary_events(payload: dict, game: Optional[int], event_id: str) -> pd.DataFrame:
    comp = summary_competition(payload)
    # Une eventos do cabeçalho e keyEvents. O uso de `or` descartava
    # keyEvents quando o cabeçalho continha apenas um gol.
    details = []
    for source_rows in [payload.get("details") or [], comp.get("details") or [], payload.get("keyEvents") or []]:
        details.extend(source_rows)

    deduped = []
    seen = set()
    for detail in details:
        key = (
            str(detail.get("id", "")),
            normalize_text(first_nonempty(nested_get(detail, ["clock", "displayValue"]), detail.get("clock"))),
            normalize_text(first_nonempty(detail.get("text"), detail.get("shortText"), detail.get("headline"))),
        )
        if key in seen:
            continue
        seen.add(key)
        deduped.append(detail)

    rows = []
    for detail in deduped:
        type_obj = detail.get("type") or {}
        team_obj = detail.get("team") or {}
        participants = detail.get("athletes") or detail.get("participants") or []
        athlete_names = []
        for item in participants:
            athlete = item.get("athlete", item) if isinstance(item, dict) else {}
            name = first_nonempty(athlete.get("displayName"), athlete.get("fullName"), athlete.get("shortName")) if isinstance(athlete, dict) else None
            if name:
                athlete_names.append(name)
        type_text = first_nonempty(type_obj.get("text"), type_obj.get("name"), type_obj.get("description"), detail.get("type"))
        rows.append({
            "jogo": game,
            "espn_event_id": event_id,
            "event_id": detail.get("id"),
            "period": safe_int(nested_get(detail, ["period", "number"]) if isinstance(detail.get("period"), dict) else detail.get("period")),
            "clock": first_nonempty(nested_get(detail, ["clock", "displayValue"]), detail.get("clock")),
            "type_text": type_text,
            "team": to_repo_team(first_nonempty(team_obj.get("displayName"), team_obj.get("name"))),
            "team_espn": first_nonempty(team_obj.get("displayName"), team_obj.get("name")),
            "player": athlete_names[0] if athlete_names else None,
            "participants_json": json.dumps(athlete_names, ensure_ascii=False),
            "text": first_nonempty(detail.get("text"), detail.get("shortText"), detail.get("headline")),
            "scoring_play": bool(detail.get("scoringPlay", False)),
            "shootout": bool(detail.get("shootout", False)),
            "penalty": bool(detail.get("penaltyKick", False)) or bool(detail.get("penalty", False)) or "penalty" in normalize_text(type_text),
            "own_goal": bool(detail.get("ownGoal", False)) or "own goal" in normalize_text(type_text),
            "yellow_card": "yellow" in normalize_text(type_text),
            "red_card": "red" in normalize_text(type_text),
            "substitution": "substitution" in normalize_text(type_text),
            "source_url": f"{ESPN_SUMMARY_URL}?event={event_id}",
            "collected_at": utc_now_iso(),
        })
    return pd.DataFrame(rows)


def parse_commentary(payload: dict, game: Optional[int], event_id: str) -> pd.DataFrame:
    rows = []
    for item in payload.get("commentary") or []:
        time_obj = item.get("time") or {}
        rows.append({
            "jogo": game,
            "espn_event_id": event_id,
            "sequence": safe_int(item.get("sequence")),
            "clock": first_nonempty(time_obj.get("displayValue"), item.get("clock")),
            "clock_seconds": safe_float(time_obj.get("value")),
            "text": item.get("text"),
            "source_url": f"{ESPN_SUMMARY_URL}?event={event_id}",
            "collected_at": utc_now_iso(),
        })
    return pd.DataFrame(rows)


def parse_team_stats(payload: dict, game: Optional[int], event_id: str) -> pd.DataFrame:
    rows = []
    for team_entry in (payload.get("boxscore") or {}).get("teams") or []:
        team_obj = team_entry.get("team") or {}
        team_espn = first_nonempty(team_obj.get("displayName"), team_obj.get("name"))
        row = {
            "jogo": game,
            "espn_event_id": event_id,
            "team_espn": team_espn,
            "team": to_repo_team(team_espn),
            "source_url": f"{ESPN_SUMMARY_URL}?event={event_id}",
            "collected_at": utc_now_iso(),
        }
        for stat in team_entry.get("statistics") or []:
            name = first_nonempty(stat.get("name"), stat.get("label"), stat.get("abbreviation"))
            if name:
                row[f"stat_{stat_name(name)}"] = first_nonempty(stat.get("value"), stat.get("displayValue"))
        rows.append(row)
    return pd.DataFrame(rows)


def parse_rosters(payload: dict, game: Optional[int], event_id: str) -> pd.DataFrame:
    rows = []
    for roster_block in payload.get("rosters") or []:
        team_obj = roster_block.get("team") or {}
        team_espn = first_nonempty(team_obj.get("displayName"), team_obj.get("name"))
        for entry in roster_block.get("roster") or roster_block.get("athletes") or []:
            athlete = entry.get("athlete") or entry
            position = athlete.get("position") or entry.get("position") or {}
            row = {
                "jogo": game,
                "espn_event_id": event_id,
                "team_espn": team_espn,
                "team": to_repo_team(team_espn),
                "athlete_id": athlete.get("id"),
                "player": first_nonempty(athlete.get("displayName"), athlete.get("fullName"), athlete.get("shortName")),
                "jersey": first_nonempty(entry.get("jersey"), athlete.get("jersey")),
                "position": first_nonempty(position.get("abbreviation"), position.get("name"), position.get("displayName")),
                "starter": entry.get("starter"),
                "active": entry.get("active"),
                "subbed_in": entry.get("subbedIn"),
                "subbed_out": entry.get("subbedOut"),
                "captain": entry.get("captain"),
                "source_url": f"{ESPN_SUMMARY_URL}?event={event_id}",
                "collected_at": utc_now_iso(),
            }
            stats = entry.get("stats") or athlete.get("stats") or []
            if isinstance(stats, dict):
                stats = [stats]
            for stat in stats:
                if isinstance(stat, dict):
                    name = first_nonempty(stat.get("name"), stat.get("label"), stat.get("abbreviation"), stat.get("displayName"))
                    if name:
                        row[f"stat_{stat_name(name)}"] = first_nonempty(stat.get("value"), stat.get("displayValue"))
            rows.append(row)
    return pd.DataFrame(rows)


def parse_officials(payload: dict, game: Optional[int], event_id: str) -> pd.DataFrame:
    rows = []
    for official in (payload.get("gameInfo") or {}).get("officials") or []:
        official_obj = official.get("official") or official
        position = official.get("position") or {}
        rows.append({
            "jogo": game,
            "espn_event_id": event_id,
            "official_id": official_obj.get("id"),
            "official": first_nonempty(official_obj.get("displayName"), official_obj.get("fullName")),
            "role": first_nonempty(position.get("displayName"), position.get("name"), official.get("type")),
            "source_url": f"{ESPN_SUMMARY_URL}?event={event_id}",
            "collected_at": utc_now_iso(),
        })
    return pd.DataFrame(rows)


def parse_penalty_shootout_payload(payload: dict, game: Optional[int], event_id: str) -> pd.DataFrame:
    rows = []
    for team_block in payload.get("shootout") or []:
        team_espn = first_nonempty(team_block.get("team"), nested_get(team_block, ["team", "displayName"]))
        for shot in team_block.get("shots") or []:
            converted = bool(shot.get("didScore"))
            rows.append({
                "jogo": game,
                "espn_event_id": event_id,
                "event_id": shot.get("id"),
                "team_id": team_block.get("id"),
                "team": to_repo_team(team_espn),
                "team_espn": team_espn,
                "player_id": shot.get("playerId"),
                "player": shot.get("player"),
                "kick_number": safe_int(shot.get("shotNumber")),
                "converted": converted,
                "result": "CONVERTED" if converted else "MISSED_OR_SAVED",
                "source_url": f"{ESPN_SUMMARY_URL}?event={event_id}",
                "collected_at": utc_now_iso(),
            })
    return pd.DataFrame(rows)


def concat_frames(items: list[pd.DataFrame]) -> pd.DataFrame:
    useful = [item for item in items if item is not None and not item.empty]
    return pd.concat(useful, ignore_index=True, sort=False) if useful else pd.DataFrame()


def collect_summaries(matches: pd.DataFrame):
    target = matches.copy()
    if ONLY_COMPLETED:
        target = target[target["completed"].fillna(False)]
    target = target.dropna(subset=["espn_event_id"])
    if MAX_SUMMARIES is not None:
        target = target.head(MAX_SUMMARIES)

    metas, events, commentary, stats, players, officials, penalties = [], [], [], [], [], [], []
    failures = []
    if FETCH_MATCH_SUMMARIES:
        for _, match in tqdm(target.iterrows(), total=len(target), desc="Resumos ESPN"):
            event_id = str(match["espn_event_id"])
            game = int(match["jogo"]) if pd.notna(match.get("jogo")) else None
            try:
                result = http.get(
                    source="espn_summary",
                    logical_name=f"summary_event_{event_id}",
                    url=ESPN_SUMMARY_URL,
                    params={"event": event_id},
                )
                payload = result.json()
                metas.append(parse_summary_meta(payload, game, event_id))
                events.append(parse_summary_events(payload, game, event_id))
                commentary.append(parse_commentary(payload, game, event_id))
                stats.append(parse_team_stats(payload, game, event_id))
                players.append(parse_rosters(payload, game, event_id))
                officials.append(parse_officials(payload, game, event_id))
                penalties.append(parse_penalty_shootout_payload(payload, game, event_id))
            except Exception as exc:
                failures.append({"jogo": game, "espn_event_id": event_id, "error": str(exc)})

    return (
        concat_frames(metas), concat_frames(events), concat_frames(commentary), concat_frames(stats),
        concat_frames(players), concat_frames(officials), concat_frames(penalties),
        pd.DataFrame(failures),
    )


summary_meta_df, events_df, commentary_df, team_stats_df, players_df, officials_df, penalties_df, summary_failures_df = collect_summaries(espn_matches)
print({
    "summary_matches": len(summary_meta_df),
    "events": len(events_df),
    "commentary": len(commentary_df),
    "team_stats": len(team_stats_df),
    "players": len(players_df),
    "officials": len(officials_df),
    "penalty_kicks": len(penalties_df),
    "summary_failures": len(summary_failures_df),
})

## 8. Conferência complementar nas páginas oficiais da FIFA

In [ ]:
def fetch_fifa_texts() -> tuple[str, list[dict]]:
    texts, meta = [], []
    for idx, url in enumerate(FIFA_OFFICIAL_URLS, start=1):
        try:
            result = http.get("fifa_official", f"official_results_page_{idx}", url)
            soup = BeautifulSoup(result.text(), "html.parser")
            for tag in soup(["script", "style", "noscript"]):
                tag.decompose()
            texts.append(" ".join(soup.stripped_strings))
            meta.append({"url": result.url, "sha256": result.sha256, "collected_at": result.collected_at})
        except Exception as exc:
            meta.append({"url": url, "error": str(exc)})
    return "\n".join(texts), meta


def variants_for_team(repo_team: str, source_name: str) -> list[str]:
    variants = set(repo_to_variants.get(repo_team, set()))
    variants.update({repo_team, source_name})
    return sorted({str(v).strip() for v in variants if str(v).strip()}, key=len, reverse=True)


def find_score_between_teams(text: str, team1_variants: list[str], team2_variants: list[str]):
    if not text:
        return None
    separators = r"\s+(\d{1,2})\s*[-–—:]\s*(\d{1,2})\s+"
    for left in team1_variants:
        for right in team2_variants:
            direct = re.compile(re.escape(left) + separators + re.escape(right), re.IGNORECASE).search(text)
            if direct:
                return int(direct.group(1)), int(direct.group(2)), "direct"
            reverse = re.compile(re.escape(right) + separators + re.escape(left), re.IGNORECASE).search(text)
            if reverse:
                return int(reverse.group(2)), int(reverse.group(1)), "reverse"
    return None


def verify_fifa(matches: pd.DataFrame, fifa_text: str) -> pd.DataFrame:
    rows = []
    for _, row in matches.iterrows():
        home, away = to_repo_team(row.get("home_team")), to_repo_team(row.get("away_team"))
        found = find_score_between_teams(
            fifa_text,
            variants_for_team(home, str(row.get("home_team_espn", ""))),
            variants_for_team(away, str(row.get("away_team_espn", ""))),
        )
        rows.append({
            "jogo": row.get("jogo"),
            "espn_event_id": str(row.get("espn_event_id", "")),
            "fifa_verified": found is not None,
            "fifa_home_score": found[0] if found else pd.NA,
            "fifa_away_score": found[1] if found else pd.NA,
            "fifa_orientation": found[2] if found else pd.NA,
            "fifa_url": FIFA_OFFICIAL_URLS[0],
        })
    return pd.DataFrame(rows)


fifa_text, fifa_fetch_meta = fetch_fifa_texts()
fifa_verification_df = verify_fifa(espn_matches, fifa_text)
print(f"Texto FIFA: {len(fifa_text):,} caracteres")
print(f"Placares localizados: {int(fifa_verification_df['fifa_verified'].sum())}")

## 9. Reconciliação independente dos resultados

In [ ]:
def derive_decision_method(row: pd.Series, event_id: str) -> str:
    detail = normalize_text(first_nonempty(row.get("status_detail"), row.get("status_description")))
    has_kicks = not penalties_df.empty and event_id in set(penalties_df["espn_event_id"].astype(str))
    if has_kicks or pd.notna(row.get("home_penalty_score")) or pd.notna(row.get("away_penalty_score")) or "pen" in detail:
        return "PENALTIES"
    if "aet" in detail or "extra time" in detail or pd.notna(row.get("home_score_120")) or pd.notna(row.get("away_score_120")):
        return "EXTRA_TIME"
    return "REGULAR_TIME"


def orient_to_calendar(row: pd.Series) -> dict:
    home, away = to_repo_team(row.get("home_team")), to_repo_team(row.get("away_team"))
    cal1 = to_repo_team(row.get("calendar_equipe1"))
    cal2 = to_repo_team(row.get("calendar_equipe2"))
    hs, a_s = safe_int(row.get("home_score")), safe_int(row.get("away_score"))
    h90, a90 = safe_int(row.get("home_score_90")), safe_int(row.get("away_score_90"))
    hp, ap = safe_int(row.get("home_penalty_score")), safe_int(row.get("away_penalty_score"))

    if normalize_text(home) == normalize_text(cal1) and normalize_text(away) == normalize_text(cal2):
        return {"equipe1": cal1, "equipe2": cal2, "gols1": hs, "gols2": a_s, "gols1_90": h90, "gols2_90": a90, "pen1": hp, "pen2": ap}
    if normalize_text(home) == normalize_text(cal2) and normalize_text(away) == normalize_text(cal1):
        return {"equipe1": cal1, "equipe2": cal2, "gols1": a_s, "gols2": hs, "gols1_90": a90, "gols2_90": h90, "pen1": ap, "pen2": hp}
    return {"equipe1": home, "equipe2": away, "gols1": hs, "gols2": a_s, "gols1_90": h90, "gols2_90": a90, "pen1": hp, "pen2": ap}


def build_validated_results(matches: pd.DataFrame) -> pd.DataFrame:
    summary_by_event = summary_meta_df.set_index("espn_event_id").to_dict("index") if not summary_meta_df.empty else {}
    fifa_by_event = fifa_verification_df.set_index("espn_event_id").to_dict("index") if not fifa_verification_df.empty else {}
    rows = []

    for _, row in matches.iterrows():
        if ONLY_COMPLETED and not bool(row.get("completed", False)):
            continue
        game = row.get("jogo")
        event_id = str(row.get("espn_event_id", ""))
        oriented = orient_to_calendar(row)
        g1, g2 = oriented["gols1"], oriented["gols2"]
        if g1 is None or g2 is None:
            continue

        summary = summary_by_event.get(event_id, {})
        summary_available = bool(summary)
        summary_pair_ok = team_pair(summary.get("summary_home_team"), summary.get("summary_away_team")) == team_pair(row.get("home_team"), row.get("away_team")) if summary_available else False
        summary_score_ok = (
            safe_int(summary.get("summary_home_score")) == safe_int(row.get("home_score"))
            and safe_int(summary.get("summary_away_score")) == safe_int(row.get("away_score"))
        ) if summary_available else False
        summary_agrees = summary_available and summary_pair_ok and summary_score_ok

        fifa = fifa_by_event.get(event_id, {})
        fifa_available = bool(fifa.get("fifa_verified", False))
        fifa_agrees = (
            safe_int(fifa.get("fifa_home_score")) == safe_int(row.get("home_score"))
            and safe_int(fifa.get("fifa_away_score")) == safe_int(row.get("away_score"))
        ) if fifa_available else False

        conflict_reasons = []
        if summary_available and not summary_agrees:
            conflict_reasons.append("ESPN_SCOREBOARD_SUMMARY_MISMATCH")
        if fifa_available and not fifa_agrees:
            conflict_reasons.append("FIFA_ESPN_SCORE_MISMATCH")

        if conflict_reasons:
            status, confidence, review = "CONFLICTING_DATA", "Baixa", True
        elif summary_agrees and fifa_agrees:
            status, confidence, review = "CONFIRMED_ESPN_SUMMARY_FIFA", "Alta", False
        elif summary_agrees:
            status, confidence, review = "CONFIRMED_ESPN_SCOREBOARD_SUMMARY", "Alta", False
        elif fifa_agrees:
            status, confidence, review = "CONFIRMED_ESPN_FIFA", "Alta", False
        else:
            status, confidence, review = "ESPN_SCOREBOARD_ONLY_REVIEW", "Media", True

        decision = derive_decision_method(row, event_id)
        pen1, pen2 = oriented["pen1"], oriented["pen2"]
        if decision == "PENALTIES" and (pen1 is None or pen2 is None) and not penalties_df.empty:
            kicks = penalties_df[penalties_df["espn_event_id"].astype(str).eq(event_id)]
            counts = kicks[kicks["converted"].fillna(False)].groupby("team").size().to_dict() if not kicks.empty else {}
            pen1, pen2 = counts.get(oriented["equipe1"], pen1), counts.get(oriented["equipe2"], pen2)

        if decision == "PENALTIES" and pen1 is not None and pen2 is not None and pen1 != pen2:
            winner = oriented["equipe1"] if pen1 > pen2 else oriented["equipe2"]
        elif g1 > g2:
            winner = oriented["equipe1"]
        elif g2 > g1:
            winner = oriented["equipe2"]
        else:
            winner = "Empate"

        rows.append({
            "jogo": int(game) if pd.notna(game) else pd.NA,
            "data": row.get("calendar_data") if pd.notna(row.get("calendar_data")) else row.get("date"),
            "fase": row.get("calendar_fase") if pd.notna(row.get("calendar_fase")) else row.get("phase_espn"),
            "equipe1": oriented["equipe1"],
            "equipe2": oriented["equipe2"],
            "gols1_real": g1,
            "gols2_real": g2,
            "placar_real": f"{g1}-{g2}",
            "gols1_90": oriented["gols1_90"],
            "gols2_90": oriented["gols2_90"],
            "vencedor_real": winner,
            "status_real": "Finalizado",
            "decision_method": decision,
            "placar_penaltis_real": f"{pen1}-{pen2}" if pen1 is not None and pen2 is not None else pd.NA,
            "vencedor_penaltis_real": winner if decision == "PENALTIES" else pd.NA,
            "fonte": row.get("source_url"),
            "source_secondary": summary.get("source_url") if summary_agrees else fifa.get("fifa_url") if fifa_agrees else pd.NA,
            "espn_event_id": event_id,
            "collected_at": row.get("collected_at"),
            "source_sha256": row.get("source_sha256"),
            "validation_status": status,
            "confidence": confidence,
            "review_required": review,
            "conflict_reason": "|".join(conflict_reasons) if conflict_reasons else pd.NA,
            "mapping_confidence": row.get("mapping_confidence"),
            "participant_change_required": bool(row.get("participant_change_required", False)),
            "scoreboard_summary_agree": summary_agrees,
            "fifa_agrees": fifa_agrees,
        })

    result = pd.DataFrame(rows)
    if not result.empty:
        result = result.sort_values("jogo", na_position="last").reset_index(drop=True)
    return result


validated_results_df = build_validated_results(espn_matches)
print(validated_results_df["validation_status"].value_counts(dropna=False))
display(validated_results_df.tail(20))

## 10. Base mestre, lacunas e conflitos

In [ ]:
def games_with_rows(frame: pd.DataFrame) -> set[int]:
    if frame.empty or "jogo" not in frame.columns:
        return set()
    return set(pd.to_numeric(frame["jogo"], errors="coerce").dropna().astype(int))


def build_master_and_gaps() -> tuple[pd.DataFrame, pd.DataFrame]:
    master = repo_matches.copy()
    if not validated_results_df.empty:
        master = master.merge(validated_results_df, on="jogo", how="left", suffixes=("_calendario", "_extraido"))

    coverage = {
        "has_summary": games_with_rows(summary_meta_df),
        "has_events": games_with_rows(events_df),
        "has_commentary": games_with_rows(commentary_df),
        "has_team_stats": games_with_rows(team_stats_df),
        "has_player_data": games_with_rows(players_df),
        "has_officials": games_with_rows(officials_df),
        "has_penalty_kicks": games_with_rows(penalties_df),
    }
    for column, game_set in coverage.items():
        master[column] = master["jogo"].astype(int).isin(game_set)

    gap_rows = []
    for _, row in master.iterrows():
        gaps = []
        if pd.isna(row.get("espn_event_id")):
            gaps.append("MATCH_NOT_MAPPED")
        if pd.isna(row.get("placar_real")):
            gaps.append("FINAL_SCORE_MISSING")
        if not bool(row.get("has_summary", False)):
            gaps.append("SUMMARY_MISSING")
        if not bool(row.get("has_events", False)):
            gaps.append("EVENTS_MISSING")
        if not bool(row.get("has_commentary", False)):
            gaps.append("COMMENTARY_MISSING")
        if not bool(row.get("has_team_stats", False)):
            gaps.append("TEAM_STATS_MISSING")
        if not bool(row.get("has_player_data", False)):
            gaps.append("PLAYER_DATA_MISSING")
        if not bool(row.get("has_officials", False)):
            gaps.append("OFFICIALS_MISSING")
        if row.get("decision_method") == "PENALTIES" and not bool(row.get("has_penalty_kicks", False)):
            gaps.append("PENALTY_KICKS_MISSING")
        if bool(row.get("review_required", False)):
            gaps.append("REVIEW_REQUIRED")
        for gap in gaps:
            gap_rows.append({
                "jogo": int(row["jogo"]),
                "data": row.get("data_calendario", row.get("data")),
                "fase": row.get("fase_calendario", row.get("fase")),
                "equipe1": row.get("equipe1_calendario", row.get("equipe1")),
                "equipe2": row.get("equipe2_calendario", row.get("equipe2")),
                "gap_type": gap,
                "validation_status": row.get("validation_status"),
            })
    return master, pd.DataFrame(gap_rows)


master_matches_df, gaps_df = build_master_and_gaps()
conflicts_df = validated_results_df[validated_results_df["validation_status"].eq("CONFLICTING_DATA")].copy() if not validated_results_df.empty else pd.DataFrame()

print("Lacunas:")
display(gaps_df["gap_type"].value_counts().rename_axis("gap").reset_index(name="quantidade") if not gaps_df.empty else pd.DataFrame({"gap": [], "quantidade": []}))

## 11. Exportação dos arquivos

In [ ]:
def build_per_match_json(master: pd.DataFrame) -> list[dict]:
    output = []
    for _, match in master.sort_values("jogo").iterrows():
        game = int(match["jogo"])
        output.append({
            "jogo": game,
            "calendar_and_result": {k: (None if pd.isna(v) else v) for k, v in match.to_dict().items()},
            "summary_meta": dataframe_records(summary_meta_df[pd.to_numeric(summary_meta_df.get("jogo"), errors="coerce").eq(game)]) if not summary_meta_df.empty else [],
            "events": dataframe_records(events_df[pd.to_numeric(events_df.get("jogo"), errors="coerce").eq(game)]) if not events_df.empty else [],
            "commentary": dataframe_records(commentary_df[pd.to_numeric(commentary_df.get("jogo"), errors="coerce").eq(game)]) if not commentary_df.empty else [],
            "team_stats": dataframe_records(team_stats_df[pd.to_numeric(team_stats_df.get("jogo"), errors="coerce").eq(game)]) if not team_stats_df.empty else [],
            "players": dataframe_records(players_df[pd.to_numeric(players_df.get("jogo"), errors="coerce").eq(game)]) if not players_df.empty else [],
            "officials": dataframe_records(officials_df[pd.to_numeric(officials_df.get("jogo"), errors="coerce").eq(game)]) if not officials_df.empty else [],
            "penalty_shootout": dataframe_records(penalties_df[pd.to_numeric(penalties_df.get("jogo"), errors="coerce").eq(game)]) if not penalties_df.empty else [],
        })
    return output


# Normalizados
write_csv(espn_matches, NORMALIZED_DIR / "espn_matches.csv")
write_csv(mapping_report, NORMALIZED_DIR / "event_mapping_report.csv")
write_csv(summary_meta_df, NORMALIZED_DIR / "espn_summary_match_meta.csv")
write_csv(events_df, NORMALIZED_DIR / "espn_match_events.csv")
write_csv(commentary_df, NORMALIZED_DIR / "espn_match_commentary.csv")
write_csv(team_stats_df, NORMALIZED_DIR / "espn_team_match_stats.csv")
write_csv(players_df, NORMALIZED_DIR / "espn_player_match_stats.csv")
write_csv(officials_df, NORMALIZED_DIR / "espn_match_officials.csv")
write_csv(penalties_df, NORMALIZED_DIR / "espn_penalty_shootouts.csv")
write_csv(fifa_verification_df, NORMALIZED_DIR / "fifa_verification.csv")

# Camada final
write_csv(master_matches_df, RUN_DIR / "wc2026_matches_master.csv")
write_csv(validated_results_df, RUN_DIR / "wc2026_results_validated.csv")
write_csv(gaps_df, REPORT_DIR / "data_gaps.csv")
write_csv(conflicts_df, REPORT_DIR / "data_conflicts.csv")
write_csv(summary_failures_df, REPORT_DIR / "summary_failures.csv")

(RUN_DIR / "wc2026_matches_full.json").write_text(
    json.dumps(build_per_match_json(master_matches_df), ensure_ascii=False, indent=2, default=str),
    encoding="utf-8",
)
(RUN_DIR / "fifa_fetch_metadata.json").write_text(json.dumps(fifa_fetch_meta, ensure_ascii=False, indent=2), encoding="utf-8")

# Patch compatível com o repositório, sem aplicá-lo automaticamente.
patch_results = validated_results_df.copy()
if not patch_results.empty:
    patch_results["placar_original"] = patch_results["equipe1"] + " " + patch_results["placar_real"] + " " + patch_results["equipe2"]
    result_columns = [
        "jogo", "data", "fase", "equipe1", "equipe2", "gols1_real", "gols2_real",
        "placar_real", "vencedor_real", "status_real", "fonte", "placar_original",
        "placar_penaltis_real", "vencedor_penaltis_real", "source_secondary",
    ]
    write_csv(patch_results[result_columns], PATCH_DIR / "resultados_reais.csv")

corrected_schedule = repo_matches.copy()
for _, row in validated_results_df.dropna(subset=["jogo"]).iterrows():
    mask = corrected_schedule["jogo"].eq(int(row["jogo"]))
    corrected_schedule.loc[mask, "equipe1"] = row["equipe1"]
    corrected_schedule.loc[mask, "equipe2"] = row["equipe2"]
    corrected_schedule.loc[mask, "confronto"] = f"{row['equipe1']} x {row['equipe2']}"
    corrected_schedule.loc[mask, "status"] = "Finalizado"
write_csv(corrected_schedule, PATCH_DIR / "matches.csv")
(PATCH_DIR / "matches.json").write_text(json.dumps(dataframe_records(corrected_schedule), ensure_ascii=False, indent=2, default=str), encoding="utf-8")

staging = pd.DataFrame({
    "jogo": validated_results_df.get("jogo"),
    "data": validated_results_df.get("data"),
    "fase": validated_results_df.get("fase"),
    "time_1": validated_results_df.get("equipe1"),
    "gols_time_1": validated_results_df.get("gols1_real"),
    "gols_time_2": validated_results_df.get("gols2_real"),
    "time_2": validated_results_df.get("equipe2"),
    "placar": validated_results_df.get("placar_real"),
    "status": "Finalizado",
    "vencedor_real": validated_results_df.get("vencedor_real"),
    "placar_penaltis_real": validated_results_df.get("placar_penaltis_real"),
    "vencedor_penaltis_real": validated_results_df.get("vencedor_penaltis_real"),
    "fonte": validated_results_df.get("fonte"),
    "source_secondary": validated_results_df.get("source_secondary"),
    "validation_status": validated_results_df.get("validation_status"),
    "review_required": validated_results_df.get("review_required"),
})
write_csv(staging, PATCH_DIR / "novos_resultados_extraidos.csv", sep=";")

print(f"Arquivos gerados em: {RUN_DIR}")

## 12. Validação de qualidade

In [ ]:
def quality_checks() -> pd.DataFrame:
    checks = []
    def add(name: str, passed: bool, detail: str, severity: str = "ERROR"):
        checks.append({"check": name, "passed": bool(passed), "severity": severity, "detail": detail})

    add("calendar_has_104_matches", len(repo_matches) == 104, f"Encontrados: {len(repo_matches)}")
    add("calendar_game_ids_unique", repo_matches["jogo"].nunique() == len(repo_matches), f"Únicos: {repo_matches['jogo'].nunique()}")
    add("scoreboard_event_ids_unique", espn_matches["espn_event_id"].nunique() == len(espn_matches), f"Eventos: {len(espn_matches)}")
    mapped = espn_matches.dropna(subset=["jogo"])
    add("mapped_game_ids_unique", mapped["jogo"].nunique() == len(mapped), f"Mapeados: {len(mapped)}")
    add("all_calendar_games_mapped", mapped["jogo"].nunique() == 104, f"Mapeados: {mapped['jogo'].nunique()}", "WARNING")
    add("validated_games_unique", validated_results_df["jogo"].nunique() == len(validated_results_df), f"Resultados: {len(validated_results_df)}")
    add("completed_scores_present", validated_results_df[["gols1_real", "gols2_real"]].notna().all(axis=None), "Placares numéricos")
    add("no_score_conflicts", conflicts_df.empty, f"Conflitos: {len(conflicts_df)}")
    add("summary_for_all_validated", games_with_rows(summary_meta_df).issuperset(games_with_rows(validated_results_df)), f"Resumos: {summary_meta_df['jogo'].nunique() if not summary_meta_df.empty else 0}", "WARNING")

    if not team_stats_df.empty:
        counts = team_stats_df.groupby("jogo").size()
        add("team_stats_max_two_rows_per_game", bool((counts <= 2).all()), f"Máximo: {int(counts.max())}", "WARNING")
    else:
        add("team_stats_available", False, "Nenhuma estatística de equipe", "WARNING")

    game98 = validated_results_df[validated_results_df["jogo"].eq(98)]
    game94 = validated_results_df[validated_results_df["jogo"].eq(94)]
    game94_ok = not game94.empty and normalize_text(game94.iloc[0]["vencedor_real"]) == normalize_text("Bélgica")
    game98_ok = not game98.empty and team_pair(game98.iloc[0]["equipe1"], game98.iloc[0]["equipe2"]) == team_pair("Espanha", "Bélgica")
    add("game_94_belgium_advances", game94_ok, "O vencedor do jogo 94 deve ser Bélgica")
    add("game_98_spain_belgium", game98_ok, "O jogo 98 deve ser Espanha x Bélgica")

    return pd.DataFrame(checks)


quality_df = quality_checks()
write_csv(quality_df, REPORT_DIR / "quality_checks.csv")
display(quality_df)

summary = {
    "run_id": RUN_ID,
    "run_at_utc": RUN_AT_UTC.isoformat(),
    "calendar_matches": int(len(repo_matches)),
    "scoreboard_events": int(len(espn_matches)),
    "mapped_games": int(espn_matches["jogo"].notna().sum()),
    "validated_results": int(len(validated_results_df)),
    "review_required": int(validated_results_df["review_required"].fillna(False).sum()) if not validated_results_df.empty else 0,
    "conflicts": int(len(conflicts_df)),
    "summary_matches": int(len(summary_meta_df)),
    "event_rows": int(len(events_df)),
    "commentary_rows": int(len(commentary_df)),
    "team_stat_rows": int(len(team_stats_df)),
    "player_rows": int(len(players_df)),
    "official_rows": int(len(officials_df)),
    "penalty_kick_rows": int(len(penalties_df)),
    "gaps": int(len(gaps_df)),
}
(REPORT_DIR / "extraction_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(summary, ensure_ascii=False, indent=2))

## 13. Resumo da execução

In [ ]:
final_overview = pd.DataFrame([
    {"indicador": "Jogos no calendário", "valor": len(repo_matches)},
    {"indicador": "Eventos no scoreboard", "valor": len(espn_matches)},
    {"indicador": "Resultados validados", "valor": len(validated_results_df)},
    {"indicador": "Resumos coletados", "valor": len(summary_meta_df)},
    {"indicador": "Eventos detalhados", "valor": len(events_df)},
    {"indicador": "Linhas de narração", "valor": len(commentary_df)},
    {"indicador": "Estatísticas de equipes", "valor": len(team_stats_df)},
    {"indicador": "Registros de jogadores", "valor": len(players_df)},
    {"indicador": "Registros de arbitragem", "valor": len(officials_df)},
    {"indicador": "Cobranças de pênaltis", "valor": len(penalties_df)},
    {"indicador": "Conflitos", "valor": len(conflicts_df)},
    {"indicador": "Lacunas", "valor": len(gaps_df)},
])
display(final_overview)

print("Principais arquivos:")
for path in [
    RUN_DIR / "wc2026_results_validated.csv",
    RUN_DIR / "wc2026_matches_master.csv",
    RUN_DIR / "wc2026_matches_full.json",
    NORMALIZED_DIR / "espn_match_events.csv",
    NORMALIZED_DIR / "espn_match_commentary.csv",
    NORMALIZED_DIR / "espn_team_match_stats.csv",
    NORMALIZED_DIR / "espn_player_match_stats.csv",
    NORMALIZED_DIR / "espn_penalty_shootouts.csv",
    REPORT_DIR / "data_gaps.csv",
    REPORT_DIR / "quality_checks.csv",
    PATCH_DIR / "resultados_reais.csv",
]:
    print(f"- {path}")

## 14. Compactar e baixar tudo em ZIP

In [ ]:
export_base = BASE_DIR / f"wc2026_extracao_independente_{RUN_ID}"
zip_file = Path(shutil.make_archive(
    str(export_base),
    "zip",
    root_dir=RUN_DIR.parent,
    base_dir=RUN_DIR.name,
))

print(f"ZIP criado: {zip_file}")
print(f"Tamanho: {zip_file.stat().st_size / (1024 * 1024):.2f} MB")

if AUTO_DOWNLOAD_ZIP:
    try:
        from google.colab import files  # type: ignore
        files.download(str(zip_file))
    except ImportError:
        print("Fora do Google Colab: use o caminho mostrado acima.")

### Uso recomendado

1. Execute as células em ordem.
2. Para um teste rápido, defina `MAX_SUMMARIES = 3`.
3. Para a extração completa, deixe `MAX_SUMMARIES = None`.
4. Revise `reports/data_gaps.csv`, `reports/data_conflicts.csv` e `reports/quality_checks.csv`.
5. Use a pasta `repository_patch/` somente após a revisão.